In [1]:
import os
import re
import json
import pickle
import subprocess
import numpy as np
import pandas as pd
import polars as pl
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import pyranges as pr
import requests
import packaging
import leidenalg
import ray

from numpy import array
from matplotlib.pyplot import rc_context

import pycisTopic
from pycisTopic.qc import get_barcodes_passing_qc_for_sample
from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
from pycisTopic.pseudobulk_peak_calling import peak_calling, export_pseudobulk
from pycisTopic.iterative_peak_calling import get_consensus_peaks
from pycisTopic.plotting.qc_plot import plot_sample_stats, plot_barcode_stats


/vf/users/sarkern2/conda/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-26 15:09:05,166	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Set directories
outDir = "outs_trial"
tmpDir = os.path.join(outDir, "temp_sbatch")

# Ensure directories exist
os.makedirs(tmpDir, exist_ok=True)

In [3]:
# Load cisTopic object
with open(os.path.join(outDir, 'cisTopicObject_merged_dbl_filtered.pkl'), 'rb') as infile:
    cistopic_obj = pickle.load(infile)
print("cisTopic object loaded successfully")

cisTopic object loaded successfully


In [4]:
print(f"meta_data shape: {cistopic_obj.cell_data.shape}")
cistopic_obj.cell_data

meta_data shape: (115346, 25)


,cisTopic_nr_frag,cisTopic_log_nr_frag,cisTopic_nr_acc,cisTopic_log_nr_acc,sample_id,barcode_rank,total_fragments_count,log10_total_fragments_count,unique_fragments_count,log10_unique_fragments_count,...,duplication_count,duplication_ratio,nucleosome_signal,tss_enrichment,pdf_values_for_tss_enrichment,pdf_values_for_fraction_of_fragments_in_peaks,pdf_values_for_duplication_ratio,barcode,Doublet_scores_fragments,Predicted_doublets_fragments
GTTGCCCGTCTAACCT-1-geriatric_01___geriatric_01,3539,3.548881,3249,3.51175,geriatric_01,3921,16886,4.227553,8417,3.925209,...,8469,0.501540,0.784517,1.718684,0.031746,0.177356,0.616556,GTTGCCCGTCTAACCT-1,0.080833,False
ACCTGTTGTGACATGC-1-geriatric_01___geriatric_01,2914,3.46449,2701,3.431525,geriatric_01,5325,13298,4.123819,6445,3.809290,...,6853,0.515341,0.664892,1.644566,0.040298,0.163479,0.465013,ACCTGTTGTGACATGC-1,0.03416,False
CTCCGGACAGTATGTT-1-geriatric_01___geriatric_01,2120,3.326336,1997,3.300378,geriatric_01,6245,10466,4.019822,5208,3.716754,...,5258,0.502389,0.841204,2.726298,0.000369,0.131631,0.465420,CTCCGGACAGTATGTT-1,0.070707,False
AAGGCCCTCCACCCTG-1-geriatric_01___geriatric_01,3523,3.546913,3226,3.508664,geriatric_01,3974,16580,4.219611,8330,3.920697,...,8250,0.497587,0.733883,2.057298,0.009449,0.176065,0.629510,AAGGCCCTCCACCCTG-1,0.046575,False
TAATCACCATTGTCAG-1-geriatric_01___geriatric_01,3684,3.56632,3376,3.528402,geriatric_01,2891,21026,4.322777,10457,4.019449,...,10569,0.502663,0.642094,1.573128,0.044495,0.107944,0.603834,TAATCACCATTGTCAG-1,0.05622,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATTCAACCAGCATGAG-1-young_08___young_08,1516,3.180699,1429,3.155032,young_08,8171,4651,3.667640,2911,3.464191,...,1740,0.374113,0.530006,2.106713,0.006262,0.009650,0.330594,ATTCAACCAGCATGAG-1,0.011013,False
AGCGCCTAGCTATATG-1-young_08___young_08,1187,3.074451,1133,3.05423,young_08,8473,3923,3.593729,2429,3.385606,...,1494,0.380831,0.568513,2.445545,0.003098,0.027961,0.272482,AGCGCCTAGCTATATG-1,0.014482,False
GACATAGAGCTGGAAA-1-young_08___young_08,1083,3.034628,1039,3.016616,young_08,8110,4366,3.640183,3012,3.478999,...,1354,0.310124,0.744953,2.022934,0.010353,0.101439,0.027013,GACATAGAGCTGGAAA-1,0.017275,False
CAATAGCTCTATCGCC-1-young_08___young_08,1160,3.064458,1123,3.05038,young_08,7861,5346,3.728110,3436,3.536180,...,1910,0.357276,0.673770,2.376238,0.004533,0.138363,0.278855,CAATAGCTCTATCGCC-1,0.065489,False


In [5]:
# Find barcodes ending in something other than -1
non_one_barcodes = cistopic_obj.cell_data.index[
    cistopic_obj.cell_data.index.str.extract(r'-([0-9]+)')[0] != '1'
]

print(f"⚠️ Found {len(non_one_barcodes)} barcodes with suffix other than -1")
print(non_one_barcodes[:10])  # show first 10


⚠️ Found 0 barcodes with suffix other than -1
Index([], dtype='object')


In [6]:
cell_data_path = "cell_data.tsv"
cell_data = pd.read_csv(cell_data_path, sep="\t", index_col=0)
print(cell_data.head())


                                   sample     sex        age celltype  \
barcode                                                                 
AAACAGCCAAGTGAAC-geriatric_5  geriatric_5  female  geriatric    MoMFs   
AAACAGCCACTTGTTC-geriatric_7  geriatric_7  female  geriatric    MoMFs   
AAACAGCCATCCAGGT-geriatric_5  geriatric_5  female  geriatric    MoMFs   
AAACAGCCATTATGGT-geriatric_7  geriatric_7  female  geriatric    MoMFs   
AAACATGCAAGCGAGC-geriatric_8  geriatric_8    male  geriatric    MoMFs   

                             celltype_sub  leiden_1  leiden_2  leiden_3  \
barcode                                                                   
AAACAGCCAAGTGAAC-geriatric_5           M1         2         5         8   
AAACAGCCACTTGTTC-geriatric_7           M2         2         5         8   
AAACAGCCATCCAGGT-geriatric_5           M2         2         5         8   
AAACAGCCATTATGGT-geriatric_7           M2         2         5         8   
AAACATGCAAGCGAGC-geriatric_8          

In [7]:
# Start from existing index
cell_data = cell_data.copy()
cell_data.reset_index(inplace=True)

# Extract barcode and sample_id from old index
cell_data[['barcode_part', 'sample_id']] = cell_data['barcode'].str.extract(r'^([A-Z0-9]+)-([a-zA-Z0-9_]+)$')

# Zero-pad sample_id (e.g. geriatric_3 → geriatric_03)
cell_data['sample_id'] = cell_data['sample_id'].apply(lambda x: re.sub(r'_(\d)$', r'_0\1', x))

# Add a fixed suffix like "-1", or customize if needed
suffix_number = '1'  # or load dynamically if available

# Build cisTopic-style index
cell_data['cisTopic_id'] = cell_data.apply(
    lambda row: f"{row['barcode_part']}-{suffix_number}-{row['sample_id']}___{row['sample_id']}", axis=1
)

# Set new index
cell_data = cell_data.set_index('cisTopic_id')
cell_data.drop(columns=['barcode', 'barcode_part'], inplace=True)

# Verify
print("✅ New cisTopic-style index:\n", cell_data.index[:5])


✅ New cisTopic-style index:
 Index(['AAACAGCCAAGTGAAC-1-geriatric_05___geriatric_05',
       'AAACAGCCACTTGTTC-1-geriatric_07___geriatric_07',
       'AAACAGCCATCCAGGT-1-geriatric_05___geriatric_05',
       'AAACAGCCATTATGGT-1-geriatric_07___geriatric_07',
       'AAACATGCAAGCGAGC-1-geriatric_08___geriatric_08'],
      dtype='object', name='cisTopic_id')


In [8]:
# Check for duplicates in cell_data
duplicates_in_cell_data = cell_data.index[cell_data.index.duplicated()]
if len(duplicates_in_cell_data) > 0:
    print(f"Duplicates in cell_data indices:\n{duplicates_in_cell_data}")

# Check for duplicates in CistopicObject cell data
duplicates_in_cistopic = cistopic_obj.cell_data.index[cistopic_obj.cell_data.index.duplicated()]
if len(duplicates_in_cistopic) > 0:
    print(f"Duplicates in CistopicObject cell data indices:\n{duplicates_in_cistopic}")
# Remove duplicates in cell_data, keeping the first occurrence
cell_data = cell_data[~cell_data.index.duplicated(keep='first')]

# Remove duplicates in CistopicObject cell data, keeping the first occurrence
cistopic_obj.cell_data = cistopic_obj.cell_data[~cistopic_obj.cell_data.index.duplicated(keep='first')]
try:
    cistopic_obj.add_cell_data(cell_data)
    print("Successfully added cleaned cell data to the CistopicObject.")
except Exception as e:
    print(f"Error while adding cell data: {e}")


Columns ['sample_id'] will be overwritten
Successfully added cleaned cell data to the CistopicObject.


In [9]:
# Step 1: Clean the cell_data by removing rows with any NaN values
cistopic_obj.cell_data = cistopic_obj.cell_data.dropna()

# Step 2: The cleaned DataFrame is now assigned back to cistopic_obj.cell_data
print("cistopic_obj.cell_data has been updated with the cleaned version.")

cistopic_obj.cell_data has been updated with the cleaned version.


In [10]:
print(cistopic_obj.cell_data.columns)

Index(['duplication_count', 'cisTopic_log_nr_frag',
       'log10_total_fragments_in_peaks_count', 'pdf_values_for_tss_enrichment',
       'barcode', 'unique_fragments_count', 'cisTopic_log_nr_acc',
       'duplication_ratio', 'unique_fragments_in_peaks_count',
       'cisTopic_nr_frag', 'barcode_rank', 'total_fragments_in_peaks_count',
       'fraction_of_fragments_in_peaks',
       'pdf_values_for_fraction_of_fragments_in_peaks',
       'Doublet_scores_fragments', 'total_fragments_count', 'cisTopic_nr_acc',
       'log10_total_fragments_count', 'log10_unique_fragments_count',
       'tss_enrichment', 'Predicted_doublets_fragments', 'nucleosome_signal',
       'pdf_values_for_duplication_ratio',
       'log10_unique_fragments_in_peaks_count', 'sample', 'sex', 'age',
       'celltype', 'celltype_sub', 'leiden_1', 'leiden_2', 'leiden_3',
       'leiden_4', 'leiden_5', 'sample_id'],
      dtype='object')


In [11]:
# Assuming 'cistopic_obj' has a DataFrame-like structure named 'cell_data'
data = cistopic_obj.cell_data

# Filter out rows with NaN values in specific columns (e.g., 'sample' and 'celltype')
filtered_data = data.dropna(subset=['duplication_count', 'unique_fragments_count',
       'unique_fragments_in_peaks_count', 'Doublet_scores_fragments',
       'total_fragments_in_peaks_count', 'fraction_of_fragments_in_peaks',
       'duplication_ratio', 'log10_unique_fragments_in_peaks_count',
       'cisTopic_log_nr_frag', 'nucleosome_signal', 'barcode_rank',
       'cisTopic_log_nr_acc', 'cisTopic_nr_frag',
       'log10_unique_fragments_count', 'tss_enrichment', 'barcode',
       'log10_total_fragments_in_peaks_count',
       'pdf_values_for_fraction_of_fragments_in_peaks',
       'pdf_values_for_duplication_ratio', 'log10_total_fragments_count',
       'Predicted_doublets_fragments', 'pdf_values_for_tss_enrichment',
       'total_fragments_count', 'cisTopic_nr_acc', 'sample', 'sex', 'age',
       'celltype', 'celltype_sub', 'leiden_1', 'leiden_2', 'leiden_3',
       'leiden_4', 'leiden_5', 'sample_id'])
# Update the cistopic_obj with the filtered data
cistopic_obj.cell_data = filtered_data

In [12]:
# Original index count (before transformation)
print("Original cell_data count:", len(cell_data))

# cisTopic cell names
cis_barcodes = set(cistopic_obj.cell_names)
print("cisTopic cell count:", len(cis_barcodes))

# Check overlap
transformed_barcodes = set(cell_data.index)
intersect = cis_barcodes.intersection(transformed_barcodes)
missing = transformed_barcodes - cis_barcodes

print(f"\n✅ Matched barcodes: {len(intersect)}")
print(f"❌ Missing barcodes after transformation: {len(missing)}")

# Optional: save or preview
print("🔍 Example of missing barcodes:")
print(list(missing)[:10])


Original cell_data count: 9393
cisTopic cell count: 115346

✅ Matched barcodes: 3035
❌ Missing barcodes after transformation: 6358
🔍 Example of missing barcodes:
['GATTGCGTCTAACTGA-1-geriatric_07___geriatric_07', 'GGTTTCCTCTTAGTCT-1-pre_ger_04___pre_ger_04', 'ATGTAAGCAGAACCGA-1-mid_age_06___mid_age_06', 'TACCGAAGTTTAACCC-1-pre_ger_08___pre_ger_08', 'TCCGGTAAGTGAAGTG-1-geriatric_03___geriatric_03', 'CTGGTCAAGATTGAGG-1-geriatric_05___geriatric_05', 'CACCGGTAGGCCTGGT-1-young_06___young_06', 'ATGACGAAGGAGGTTA-1-geriatric_04___geriatric_04', 'AAGGCCCTCCTAAGAC-1-geriatric_05___geriatric_05', 'TCCGGTTTCCGCACAA-1-old_03___old_03']


In [13]:
from collections import Counter

# Extract sample names from cisTopic cell names
sample_names = [x.split('-')[2].split('___')[0] for x in cistopic_obj.cell_names]

# Count how many cells per sample
sample_counts = Counter(sample_names)

# Print sample set and counts
print("✅ Samples in cisTopic object:")
for sample, count in sample_counts.items():
    print(f"{sample}: {count} cells")

print(f"\nTotal unique samples: {len(sample_counts)}")
print(f"Total cells: {sum(sample_counts.values())}")


✅ Samples in cisTopic object:
geriatric_01: 2229 cells
geriatric_02: 2959 cells
geriatric_03: 2956 cells
geriatric_04: 3188 cells
geriatric_05: 4792 cells
geriatric_06: 2713 cells
geriatric_07: 2510 cells
geriatric_08: 2712 cells
mid_age_01: 2783 cells
mid_age_02: 2067 cells
mid_age_03: 2548 cells
mid_age_04: 1018 cells
mid_age_05: 3478 cells
mid_age_06: 3240 cells
mid_age_07: 3640 cells
mid_age_08: 3594 cells
old_01: 4128 cells
old_02: 4202 cells
old_03: 2088 cells
old_04: 3770 cells
old_05: 3709 cells
old_06: 3206 cells
old_07: 2637 cells
old_08: 3421 cells
pre_ger_01: 2355 cells
pre_ger_02: 2390 cells
pre_ger_03: 2535 cells
pre_ger_04: 3253 cells
pre_ger_05: 2223 cells
pre_ger_06: 1785 cells
pre_ger_07: 4225 cells
pre_ger_08: 2774 cells
young_01: 2817 cells
young_02: 2348 cells
young_03: 2547 cells
young_04: 1896 cells
young_05: 3109 cells
young_06: 2794 cells
young_07: 2727 cells
young_08: 1980 cells

Total unique samples: 40
Total cells: 115346


In [14]:
# Alternatively, you can check if there are any NaN values in the entire DataFrame
any_nans = cistopic_obj.cell_data.isna().any().any()
print(f"Are there any NaN values in the DataFrame? {any_nans}")

Are there any NaN values in the DataFrame? False


In [15]:
print(f"meta_data shape: {cistopic_obj.cell_data.shape}")

meta_data shape: (3035, 35)


In [16]:
# Save
# Save cisTopic object correctly using os.path.join
with open(os.path.join(outDir, 'cisTopicObject_filtered_annotated.pkl'), 'wb') as f:
    pickle.dump(cistopic_obj, f)

print("✅ cisTopic object saved successfully.")

✅ cisTopic object saved successfully.


In [17]:
# Load cisTopic object
with open(os.path.join(outDir, 'cisTopicObject_filtered_annotated.pkl'), 'rb') as infile:
    cistopic_obj = pickle.load(infile)
print("cisTopic object loaded successfully")

cisTopic object loaded successfully


In [18]:
print(f"meta_data shape: {cistopic_obj.cell_data.shape}")
cistopic_obj.cell_data

meta_data shape: (3035, 35)


,duplication_count,cisTopic_log_nr_frag,log10_total_fragments_in_peaks_count,pdf_values_for_tss_enrichment,barcode,unique_fragments_count,cisTopic_log_nr_acc,duplication_ratio,unique_fragments_in_peaks_count,cisTopic_nr_frag,...,sex,age,celltype,celltype_sub,leiden_1,leiden_2,leiden_3,leiden_4,leiden_5,sample_id
GTCATTAAGTATTGCA-1-geriatric_01___geriatric_01,2789,3.025306,3.290480,0.017028,GTCATTAAGTATTGCA-1,3188,3.006894,0.466622,1027,1060,...,male,geriatric,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,geriatric_01
GTAAGCTTCACAGACT-1-geriatric_01___geriatric_01,12168,3.670153,3.934094,0.041972,GTAAGCTTCACAGACT-1,13118,3.638789,0.481215,4488,4679,...,male,geriatric,MoMFs,M1,2.0,5.0,8.0,10.0,11.0,geriatric_01
GTATCGCCATGTCGCG-1-geriatric_01___geriatric_01,9766,3.685652,3.967408,0.035460,GTATCGCCATGTCGCG-1,9559,3.649335,0.505356,4662,4849,...,male,geriatric,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,geriatric_01
ATCGAGGCAGCCAGTT-1-geriatric_01___geriatric_01,12050,3.806384,4.089764,0.004402,ATCGAGGCAGCCAGTT-1,12081,3.757092,0.499358,6180,6403,...,male,geriatric,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,geriatric_01
CGTTGCAAGCTTTGTT-1-geriatric_01___geriatric_01,4554,3.10721,3.388456,0.010438,CGTTGCAAGCTTTGTT-1,4682,3.08849,0.493071,1232,1280,...,male,geriatric,MoMFs,M1,2.0,5.0,8.0,10.0,11.0,geriatric_01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACATAGCTCCTCCATA-1-young_08___young_08,2264,3.362105,3.561101,0.022667,ACATAGCTCCTCCATA-1,3692,3.328787,0.380121,2211,2302,...,male,young,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,young_08
GTTACAGGTCGTTACT-1-young_08___young_08,2379,3.27485,3.485863,0.003779,GTTACAGGTCGTTACT-1,3651,3.246006,0.394527,1816,1883,...,male,young,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,young_08
TATCCAGCATTGCAGC-1-young_08___young_08,3072,3.27738,3.474944,0.018322,TATCCAGCATTGCAGC-1,4849,3.257198,0.387830,1830,1894,...,male,young,MoMFs,M1,2.0,5.0,8.0,10.0,11.0,young_08
TTTGGCTGTCCCGAAG-1-young_08___young_08,1519,3.170555,3.364551,0.015940,TTTGGCTGTCCCGAAG-1,2516,3.156246,0.376456,1433,1481,...,male,young,MoMFs,M2,2.0,5.0,8.0,10.0,11.0,young_08
